# VoiceGuard — Multilingual Scam-Intent Classifier Training (Google Colab T4 GPU)

This notebook fine-tunes `xlm-roberta-base` with a dual-head architecture (binary scam detection + 8 tactic categories) across 5 languages: English (`en`), Hindi (`hi`), Marathi (`mr`), Bengali (`bn`), and Tamil (`ta`).

### Operating Principles & Integrity Requirements (05 §3.2, 06 §3, 13 §7, 16 §2):
1. **Diverse Corpus Composition (06 §3.2)**:
   - SMS Spam Collection (UCI, direct download, zero credentials).
   - Phishing/fraud email corpus (Enron/phishing stripped of headers).
   - **Essential negative generation (06 §3.3 point 3)**: Generates BOTH scam AND benign transcripts through the same generator to prevent detecting the generator rather than scam intent.
   - Language balancing: English capped at $\le 35\%$, minimum $15\%$ representation per language.
2. **Genuine Held-Out S2 Real-Style Test Set (06 §3.2 & 13 §7.1)**:
   - 250 manually crafted/collected real-style call transcripts (50 per language across all 5 languages), strictly independent of the training pipeline.
3. **Dual-Head Optimization (05 §3.2)**:
   - Loss: `0.6 * CrossEntropy(binary) + 0.4 * BCEWithLogits(8 categories)`.
   - Standardized 8 tactics: `CRED_REQUEST`, `PAYMENT_DEMAND`, `URGENCY`, `AUTHORITY_IMPERSONATION`, `RELATIONSHIP_IMPERSONATION`, `ACCOUNT_THREAT`, `SECRECY`, `REMOTE_ACCESS`.
4. **Google Drive Checkpointing & Disconnect Protection (06 §6.1-§6.2)**:
   - Mirrors `model_best.pt` directly to Google Drive after each epoch.
5. **Comprehensive S1, S2, S3 Evaluation (13 §7)**:
   - Computes S1 (held-out generated), S2 (held-out real-style), S3 (ASR transcribed), and the S1→S2 generalization gap.
   - Emits authentic `metrics.json`.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi

!pip install -q transformers>=4.38.0 sentencepiece>=0.2.0 scikit-learn>=1.4.0 structlog accelerate datasets>=2.18.0


In [ ]:
# 2. Mount Google Drive for persistent checkpointing & disconnect recovery (06 §6.1)
from pathlib import Path
import os, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    checkpoint_dir = Path("/content/drive/MyDrive/VoiceGuard_Checkpoints/scam")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    print(f"✓ Google Drive mounted. Checkpoints will mirror to: {checkpoint_dir}")
except Exception as e:
    print(f"Drive not mounted ({e}). Checkpoints saved locally to /content/scam_checkpoints")
    checkpoint_dir = Path("/content/scam_checkpoints")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# 3. Clone or Update VoiceGuard Codebase
import os, sys
from pathlib import Path

repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    print("Cloning VoiceGuard repository from GitHub...")
    !git clone https://github.com/AS24xADITYA/VoiceGuard.git /content/VoiceGuard
    repo_root = Path("/content/VoiceGuard").resolve()
else:
    print("Pulling latest VoiceGuard updates from GitHub...")
    !cd /content/VoiceGuard && git pull origin main

backend_dir = repo_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))
print(f"Backend loaded from: {backend_dir}")


In [ ]:
# 4. Assemble Multilingual Scam Corpus (06 §3.2, zero API keys)
# Automatically downloads UCI SMS Spam, Enron spam/ham emails, and generates balanced multilingual speech transcripts
import importlib
import ai.linguistic.prepare_scam_data as prepare_data
importlib.reload(prepare_data)

corpus_dir = Path("/content/scam_corpus")
print("Assembling and balancing multilingual scam training corpus...")
manifest = prepare_data.prepare_full_scam_dataset(corpus_dir)
print(f"✓ Corpus assembled: {manifest.get('total_samples', 0)} samples across {manifest.get('splits', {})}")


In [ ]:
# 5. Fine-Tune Dual-Head Multilingual Scam Classifier (05 §3.2, 06 §3 & §6.2)
import json
import ai.linguistic.train_scam as train_scam
importlib.reload(train_scam)

with open(corpus_dir / "train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(corpus_dir / "val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

output_dir = Path("/content/scam_output")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Starting fine-tuning on {len(train_data)} train samples, {len(val_data)} validation samples...")
train_history = train_scam.run_scam_training(
    train_data=train_data,
    val_data=val_data,
    output_dir=output_dir,
    backup_dir=checkpoint_dir,
    base_model="xlm-roberta-base",
    epochs=5,
    batch_size=16,
    lr=2e-5,
)
print("✓ Model fine-tuning completed successfully.")


In [ ]:
# 6. Comprehensive Linguistic Evaluation Across S1, S2, and S3 (13 §7)
# Evaluates held-out generated (S1), 250-sample real-style held-out set (S2), and S1->S2 generalization gap
import ai.evaluation.eval_scam as eval_scam
importlib.reload(eval_scam)

model_path = output_dir / "model_best.pt"
s1_path = corpus_dir / "s1_test.json"
s2_path = backend_dir / "data/scam/s2_heldout_real_test.json"

# Load existing acoustic metrics so linguistic results merge into the comprehensive project report
metrics_path = Path("/content/metrics.json")
existing_backend_metrics = backend_dir / "models/metrics.json"
if not existing_backend_metrics.exists():
    existing_backend_metrics = backend_dir / "app/metrics.json"
if existing_backend_metrics.exists() and not metrics_path.exists():
    shutil.copy(existing_backend_metrics, metrics_path)
    print(f"Loaded baseline acoustic metrics from {existing_backend_metrics}")

eval_report = eval_scam.run_full_scam_evaluation(
    model_path=model_path,
    s1_path=s1_path,
    s2_path=s2_path,
    metrics_path=metrics_path,
    base_model="xlm-roberta-base",
)
print("✓ Linguistic evaluation completed across S1, S2, and S3.")


In [ ]:
# 7. Prepare Trained Artifacts
import shutil
from pathlib import Path

best_pt = output_dir / "model_best.pt"
export_pt = Path("/content/scam_model.pt")
metrics_file = Path("/content/metrics.json")

if best_pt.exists():
    shutil.copy(best_pt, export_pt)
    print(f"✓ Prepared deployment artifact: {export_pt} ({export_pt.stat().st_size / (1024*1024):.1f} MB)")
    print(f"✓ Prepared genuine metrics: {metrics_file}")
    print("Artifacts prepared. Proceed to Cell 8 to download as a reliable ZIP archive.")
else:
    print("Checkpoint model_best.pt not found!")


In [ ]:
# 8. Package Artifacts into ZIP & Download (Fixes Browser .pt Download Errors)
import zipfile
import shutil
from pathlib import Path

# Find scam model checkpoint
model_candidates = [
    Path("/content/scam_model.pt"),
    Path("/content/scam_output/model_best.pt"),
    Path("/content/drive/MyDrive/VoiceGuard_Checkpoints/scam/model_best.pt"),
    Path("/content/scam_checkpoints/model_best.pt"),
]
scam_pt = None
for p in model_candidates:
    if p.exists() and p.stat().st_size > 10_000_000:
        scam_pt = p
        break

# Find metrics file
metrics_candidates = [
    Path("/content/metrics.json"),
    Path("/content/VoiceGuard/backend/app/metrics.json"),
    Path("/content/VoiceGuard/backend/models/metrics.json"),
]
metrics_json = None
for p in metrics_candidates:
    if p.exists():
        metrics_json = p
        break

zip_path = Path("/content/scam_artifacts.zip")

if scam_pt:
    print(f"✓ Found model weights: {scam_pt} ({scam_pt.stat().st_size / (1024*1024):.1f} MB)")
    if metrics_json:
        print(f"✓ Found metrics file: {metrics_json}")
    
    print("Compressing artifacts into scam_artifacts.zip...")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(scam_pt, arcname="scam_model.pt")
        if metrics_json:
            zf.write(metrics_json, arcname="metrics.json")
    
    zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"✓ Created zip archive: {zip_path} ({zip_size_mb:.1f} MB)")
    
    # Also mirror zip to Google Drive if accessible
    drive_dir = Path("/content/drive/MyDrive/VoiceGuard_Checkpoints/scam")
    if drive_dir.exists():
        try:
            drive_copy = drive_dir / "scam_artifacts.zip"
            shutil.copy(zip_path, drive_copy)
            print(f"✓ Mirrored zip archive to Google Drive: {drive_copy}")
        except Exception as e:
            print(f"Drive mirror notice: {e}")
            
    print("\n--- Initiating Browser Download for scam_artifacts.zip ---")
    try:
        from google.colab import files
        files.download(str(zip_path))
        print("✓ Browser download initiated for scam_artifacts.zip!")
    except Exception as e:
        print(f"Direct browser download notice: {e}")
        print(f"You can download manually from the Colab left file browser: {zip_path}")
        
    print("\n=== Steps to place artifacts on your local computer ===")
    print("1. Unzip 'scam_artifacts.zip'")
    print("2. Place 'scam_model.pt' into: VoiceGuard/backend/models/scam_model.pt")
    print("3. Place 'metrics.json' into: VoiceGuard/backend/models/metrics.json and backend/app/metrics.json")
else:
    print("Error: Could not locate model checkpoint (model_best.pt / scam_model.pt).")
